# Backpropagation: how it runs
Part 1 trains the 2-2-1 regression network (linear activations, MSE) with our own code, then with Keras from the
same starting weights. Part 2 does the same for classification (sigmoid activations, binary cross-entropy).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

keras.utils.set_random_seed(0)

## Part 1: regression

In [ ]:
df = pd.DataFrame([[8, 8, 4], [7, 9, 5], [6, 10, 6], [5, 12, 7]], columns=["cgpa", "profile_score", "lpa"])
X, Y = df[["cgpa", "profile_score"]].to_numpy(float), df["lpa"].to_numpy(float)

def initialize_parameters(layer_dims):
    # all weights 0.1, all biases 0; W[l] has shape (nodes in layer l-1, nodes in layer l)
    return {l: [np.full((layer_dims[l - 1], layer_dims[l]), 0.1), np.zeros(layer_dims[l])]
            for l in range(1, len(layer_dims))}

def forward_regression(x, params):
    a1 = params[1][0].T @ x + params[1][1]        # hidden outputs O11, O12 (linear)
    y_hat = params[2][0].T @ a1 + params[2][1]    # output (linear), an array of length 1
    return y_hat[0], a1

def update_regression(params, x, y, y_hat, a1, lr):
    g = -2 * (y - y_hat)                          # dL/dy_hat
    W2 = params[2][0][:, 0].copy()               # old output weights, needed for layer 1
    params[2][0][:, 0] -= lr * g * a1             # W2_11, W2_21
    params[2][1][0] -= lr * g                     # b21
    params[1][0] -= lr * np.outer(x, g * W2)      # all four W1
    params[1][1] -= lr * g * W2                   # b11, b12

def train_regression(epochs=5, lr=0.001):
    params, history = initialize_parameters([2, 2, 1]), []
    for epoch in range(epochs):
        losses = []
        for x, y in zip(X, Y):                    # one student at a time, in order
            y_hat, a1 = forward_regression(x, params)
            losses.append((y - y_hat) ** 2)
            update_regression(params, x, y, y_hat, a1, lr)
        history.append(np.mean(losses))           # average loss of the epoch
    return params, history

params, hist_reg = train_regression(epochs=5)
print("average loss per epoch:", np.round(hist_reg, 3))
print("W1:\n", params[1][0].round(4), "\nb1:", params[1][1].round(4))
print("W2:", params[2][0].ravel().round(4), "b2:", params[2][1].round(4))
print("predictions:", [round(forward_regression(x, params)[0], 2) for x in X])

### The same training in Keras
Same architecture, same starting weights, plain SGD with learning rate 0.001, one row per update (batch_size=1),
rows in the original order (shuffle=False).

In [ ]:
def keras_model(activation, loss, lr):
    model = keras.Sequential([keras.Input(shape=(2,)),
                              keras.layers.Dense(2, activation=activation),
                              keras.layers.Dense(1, activation=activation)])
    model.set_weights([np.full((2, 2), 0.1), np.zeros(2), np.full((2, 1), 0.1), np.zeros(1)])
    model.compile(optimizer=keras.optimizers.SGD(learning_rate=lr), loss=loss)
    return model

reg = keras_model("linear", "mse", 0.001)
reg.summary()
h = reg.fit(X, Y, epochs=5, batch_size=1, shuffle=False, verbose=0)
print("Keras loss per epoch:", np.round(h.history["loss"], 3))
print("Keras weights:", [w.round(4).ravel() for w in reg.get_weights()])

In [ ]:
# a longer run: 75 epochs
_, hist_reg75 = train_regression(epochs=75)
print("epoch 75 average loss:", round(hist_reg75[-1], 4))
pd.DataFrame({"epoch": range(1, 76), "ours": hist_reg75,
              "keras": keras_model("linear", "mse", 0.001).fit(X, Y, epochs=75, batch_size=1, shuffle=False,
                                                                   verbose=0).history["loss"]}
             ).to_csv("data/regression_loss.csv", index=False)

## Part 2: classification

In [ ]:
dfc = pd.DataFrame([[8, 8, 1], [7, 9, 1], [6, 10, 0], [5, 5, 0]], columns=["cgpa", "profile_score", "placed"])
Xc, Yc = dfc[["cgpa", "profile_score"]].to_numpy(float), dfc["placed"].to_numpy(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward_classification(x, params):
    a1 = sigmoid(params[1][0].T @ x + params[1][1])        # O11, O12
    y_hat = sigmoid(params[2][0].T @ a1 + params[2][1])    # probability of placement
    return y_hat[0], a1

def gradients_classification(params, x, y, y_hat, a1):
    d_out = -(y - y_hat)                         # dL/dz_final: sigmoid and log cancel
    W2 = params[2][0][:, 0]
    d_hidden = d_out * W2 * a1 * (1 - a1)        # dL/dz of each hidden node
    return {"W2": d_out * a1, "b2": d_out, "W1": np.outer(x, d_hidden), "b1": d_hidden}

def bce(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

p0 = initialize_parameters([2, 2, 1])
y_hat, a1 = forward_classification(Xc[0], p0)
print("student 1: O11, O12 =", a1.round(4), "y_hat =", round(y_hat, 4), "loss =", round(bce(Yc[0], y_hat), 4))
gc = gradients_classification(p0, Xc[0], Yc[0], y_hat, a1)
for k, v in gc.items():
    print(k, np.round(v, 5))

### Check the classification gradients with GradientTape

In [ ]:
tv = {"W1": tf.Variable(p0[1][0]), "b1": tf.Variable(p0[1][1]),
      "W2": tf.Variable(p0[2][0][:, 0]), "b2": tf.Variable(p0[2][1][0])}
with tf.GradientTape() as tape:
    a1_tf = tf.sigmoid(tf.linalg.matvec(tf.transpose(tv["W1"]), Xc[0]) + tv["b1"])
    y_tf = tf.sigmoid(tf.reduce_sum(tv["W2"] * a1_tf) + tv["b2"])
    loss_tf = -(Yc[0] * tf.math.log(y_tf) + (1 - Yc[0]) * tf.math.log(1 - y_tf))
tg = tape.gradient(loss_tf, tv)
print({k: bool(np.allclose(tg[k].numpy(), gc[k])) for k in gc})

### Training with our code and with Keras

In [ ]:
def train_classification(epochs, lr):
    params, history = initialize_parameters([2, 2, 1]), []
    for epoch in range(epochs):
        losses = []
        for x, y in zip(Xc, Yc):
            y_hat, a1 = forward_classification(x, params)
            losses.append(bce(y, y_hat))
            g = gradients_classification(params, x, y, y_hat, a1)
            params[2][0][:, 0] -= lr * g["W2"]; params[2][1][0] -= lr * g["b2"]
            params[1][0] -= lr * g["W1"]; params[1][1] -= lr * g["b1"]
        history.append(np.mean(losses))
    return params, history

pc, hist_cls = train_classification(epochs=50, lr=0.001)
print("ours, epochs 1, 2, 50:", np.round([hist_cls[0], hist_cls[1], hist_cls[-1]], 4))
cls = keras_model("sigmoid", "binary_crossentropy", 0.001)
hk = cls.fit(Xc, Yc, epochs=50, batch_size=1, shuffle=False, verbose=0).history["loss"]
print("Keras, epochs 1, 2, 50:", np.round([hk[0], hk[1], hk[-1]], 4))
print("predictions:", [round(forward_classification(x, pc)[0], 3) for x in Xc])
pd.DataFrame({"epoch": range(1, 51), "ours": hist_cls, "keras": hk}).to_csv("data/classification_loss.csv", index=False)

### Why it is stuck near 0.69, and what changes it
0.693 = log 2 is the loss of always predicting 0.5. Two students are placed and two are not, and the start makes
both hidden nodes identical. A bigger learning rate and many more epochs let it move.

In [ ]:
print("log 2 =", round(np.log(2), 4))
for lr, ep in [(0.001, 2000), (0.1, 2000), (0.5, 2000)]:
    pc2, hh = train_classification(epochs=ep, lr=lr)
    print(f"lr {lr}, {ep} epochs: loss {hh[-1]:.4f}, predictions",
          [round(forward_classification(x, pc2)[0], 2) for x in Xc], "W1 columns equal:",
          bool(np.allclose(pc2[1][0][:, 0], pc2[1][0][:, 1])))

In [ ]:
# too big a step? with lr 0.5 the epoch loss goes up as well as down
for lr in [0.1, 0.5]:
    _, hh = train_classification(epochs=2000, lr=lr)
    hh = np.array(hh)
    print(f"lr {lr}: final {hh[-1]:.3f}, lowest {hh.min():.3f} at epoch {hh.argmin() + 1}, epochs where the loss rose: {(np.diff(hh) > 0).sum()}")
